In [1]:
import os
import re
import json
import shutil
import random
import warnings
import numpy as np
import pandas as pd
import unicodedata

import os
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, log_loss, classification_report


import nltk
from nltk.corpus import stopwords

from collections import Counter
from joblib import dump, load

from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer as SklearnTfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    log_loss,
    confusion_matrix
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.utils import shuffle


import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

from datasets import Dataset as HFDataset




warnings.filterwarnings("ignore")

nltk.download("stopwords")
STOPWORD = set(stopwords.words("french"))

RANDOM_STATE = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [2]:
print(DEVICE)

cuda


Chargement données


In [3]:
def load_pres(fname, for_train=True):
    """
    Lecture fichier
    Retour (labels, texts) si for_train=True sinon texts
    """
    texts, labels = [], []
    if for_train:
        pattern = re.compile(r"^<[0-9]+:[0-9]+:([CM])>(.*)$")
    else:
        pattern = re.compile(r"^<[0-9]+:[0-9]+>(.*)$")

    with open(fname, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if len(line) < 3:
                continue

            match = pattern.match(line)
            if not match:
                continue

            if for_train:
                label, text = match.group(1), match.group(2).strip()
                labels.append(1 if label == "M" else 0)  # 1=Mitterrand, 0=Chirac
            else:
                text = match.group(1).strip()

            texts.append(text)

    return (np.array(labels), texts) if for_train else texts

In [4]:
train_labels, train_texts = load_pres("/content/corpus.tache1.learn.utf8")
test_texts = load_pres("/content/corpus.tache1.test.utf8", for_train=False)

Classe Transformer

In [5]:
class TransformerWrapper:
    def __init__(
        self,
        model_name="distilbert-base-multilingual-cased",
        max_len=64,
        batch_size=4,
        lr=1.5e-5,
        epochs=6,
        output_dir="./results_transformer"
    ):
        self.model_name = model_name
        self.max_len = max_len
        self.batch_size = batch_size
        self.lr = lr
        self.epochs = epochs
        self.output_dir = output_dir

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=2,
            low_cpu_mem_usage=True,
            device_map = "auto"
        )

        self.trainer = None

    def _tokenize_function(self, examples):
        return self.tokenizer(
            examples["text"],
            truncation=True,
            max_length=self.max_len
        )

    def fit(self, texts, y, texts_valid=None, y_valid=None):
        train_ds = HFDataset.from_dict({
            "text": list(texts),
            "labels": list(map(int, y))
        })
        train_ds = train_ds.map(
            self._tokenize_function,
            batched=True,
            remove_columns=["text"]
        )

        eval_ds = None
        if texts_valid is not None and y_valid is not None:
            eval_ds = HFDataset.from_dict({
                "text": list(texts_valid),
                "labels": list(map(int, y_valid))
            })
            eval_ds = eval_ds.map(
                self._tokenize_function,
                batched=True,
                remove_columns=["text"]
            )

        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

        y_arr = np.array(list(map(int, y)))
        class_counts = np.bincount(y_arr, minlength=2)

        # poids équilibrés + léger bonus pour la classe 1
        class_weights = class_counts.sum() / (len(class_counts) * np.maximum(class_counts, 1))
        class_weights = class_weights.astype(np.float32)
        class_weights[1] *= 2.0
        class_weights = torch.tensor(class_weights, dtype=torch.float)

        steps_per_epoch = max(1, len(train_ds) // self.batch_size)
        warmup_steps = max(1, int(steps_per_epoch * self.epochs * 0.1))

        args = TrainingArguments(
            output_dir=self.output_dir,
            num_train_epochs=self.epochs,
            learning_rate=self.lr,
            per_device_train_batch_size=self.batch_size,
            per_device_eval_batch_size=self.batch_size,
            gradient_accumulation_steps=1,
            weight_decay=0.01,
            warmup_steps=warmup_steps,
            lr_scheduler_type="linear",
            logging_strategy="epoch",
            eval_strategy="epoch" if eval_ds is not None else "no",
            save_strategy="epoch" if eval_ds is not None else "no",
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            greater_is_better=True if eval_ds is not None else None,
            report_to="none",
            dataloader_num_workers=2,
            dataloader_pin_memory=False,
            fp16=True,
            bf16=False,
            remove_unused_columns=True,
            save_total_limit=1,
            seed=RANDOM_STATE,
            label_smoothing_factor=0.05
        )

        callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]

        self.trainer = WeightedTrainer(
            model=self.model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=eval_ds,
            processing_class=self.tokenizer,
            data_collator=data_collator,
            compute_metrics=hf_compute_metrics,
            callbacks=callbacks,
            class_weights=class_weights
        )

        self.trainer.train()
        self.model = self.trainer.model
        self.trainer.save_model(self.output_dir)
        self.tokenizer.save_pretrained(self.output_dir)
        return self

    def predict_proba(self, texts):
        ds = HFDataset.from_dict({"text": list(texts)})
        ds = ds.map(
            self._tokenize_function,
            batched=True,
            remove_columns=["text"]
        )

        if self.trainer is not None:
            preds = self.trainer.predict(ds)
            logits = preds.predictions
            probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
            return probs

        self.model.eval()

        local_device = torch.device(DEVICE)
        self.model.to(local_device)

        enc = self.tokenizer(
            list(texts),
            truncation=True,
            padding=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        probs = []
        bs = self.batch_size

        with torch.no_grad():
            n = enc["input_ids"].size(0)
            for i in range(0, n, bs):
                batch = {k: v[i:i+bs].to(local_device) for k, v in enc.items()}
                outputs = self.model(**batch)
                p = torch.softmax(outputs.logits, dim=1).cpu().numpy()
                probs.append(p)

        return np.vstack(probs)

    def predict(self, texts):
        probs = self.predict_proba(texts)
        return np.argmax(probs, axis=1)

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        conf = model.module.config if hasattr(model, "module") else model.config

        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device)
        )
        loss = loss_fct(logits.view(-1, conf.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss


def hf_compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
    }

In [7]:
# mitt_sentences = load_mitterrand()
with open("/content/mitterrand.txt", "r", encoding="utf-8") as f:
    text = f.read()


text = re.sub(r"(?m)^-\s*", "", text)
text = text.replace("\ufeff"," ")
text = text.replace("\n"," ")


mitt_sentences = re.findall(r'[^.!?]+[.!?]+', text)

mitt_sentences = [s.strip() for s in mitt_sentences if s.strip()]
# mitt_sentences = preprocess_mitterrand(mitt_sentences)

mitt_labels = np.ones(len(mitt_sentences), dtype=int)


Avant l'augmentation de données, on garde 10% du dataset de base pour effectuer des tests plus tard

In [8]:
X_train_split, X_test, y_train_split, y_test = train_test_split(
    train_texts, train_labels, test_size=0.3, random_state=42, stratify=train_labels, shuffle=True
)

In [9]:
final_texts = np.concatenate([X_train_split,mitt_sentences])
final_labels = np.concatenate([y_train_split, mitt_labels])

print(f"Train : {len(final_texts)} phrases")
print("Répartition des classes :", Counter(final_labels))
print("0 = Chirac, 1 = Mitterrand")


Train : 74150 phrases
Répartition des classes : Counter({np.int64(0): 44900, np.int64(1): 29250})
0 = Chirac, 1 = Mitterrand


In [34]:
mots_par_phrase = [len(phrase.split()) for phrase in final_texts]
print("max :", np.max(mots_par_phrase))
moyenne = sum(mots_par_phrase) / len(mots_par_phrase)
print(f"Moyenne de mots : {moyenne:.2f}")

max : 386
Moyenne de mots : 21.62


In [28]:


train_cam, y_train_cam = shuffle(final_texts, final_labels, random_state=42)

test_cam, y_test_cam = X_test, y_test

train_cam_dataset = Dataset.from_dict({"text": train_cam, "label": y_train_cam})
test_cam_dataset = Dataset.from_dict({"text": test_cam, "label": y_test_cam})

In [37]:
trainer = TransformerWrapper(
    model_name="camembert/camembert-large",
    max_len=128,
    batch_size=16,
    lr=1e-5,
    epochs=3
)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

CamembertForSequenceClassification LOAD REPORT from: camembert/camembert-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [38]:
trainer.fit(
    texts=train_cam_dataset["text"],
    y=train_cam_dataset["label"],
    texts_valid=test_cam_dataset["text"],
    y_valid=test_cam_dataset["label"]
)

Map:   0%|          | 0/74150 [00:00<?, ? examples/s]

Map:   0%|          | 0/5742 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 6, 'bos_token_id': 5}.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.344933,0.322672,0.895333,0.661408,0.573803,0.780585
2,0.214093,0.374762,0.922675,0.715019,0.691067,0.740691
3,0.145697,0.450913,0.925636,0.727852,0.698898,0.759309


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
preds_test = trainer.predict(X_test)
print("\n", classification_report(y_test, preds_test, digits=4))


Map:   0%|          | 0/5742 [00:00<?, ? examples/s]


               precision    recall  f1-score   support

           0     0.9647    0.9265    0.9452      4990
           1     0.6137    0.7753    0.6851       752

    accuracy                         0.9067      5742
   macro avg     0.7892    0.8509    0.8151      5742
weighted avg     0.9188    0.9067    0.9111      5742



In [39]:
preds = trainer.predict_proba(test_texts)

proba_class_1 = preds[:, 1]

np.savetxt("testing.csv", proba_class_1, fmt="%.8f")


Map:   0%|          | 0/27162 [00:00<?, ? examples/s]

In [40]:
print(f"Meilleur checkpoint chargé : {trainer.trainer.state.best_model_checkpoint}")
print(f"Meilleur score F1 : {trainer.trainer.state.best_metric}")

Meilleur checkpoint chargé : ./results_transformer/checkpoint-13905
Meilleur score F1 : 0.7278521351179095
